<a href="https://colab.research.google.com/github/MJMortensonWarwick/AnalyticsEngineering2026/blob/main/3_01_testing_and_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 3.01 Testing and Git: From notebook to trustworthy code

| Part | Time | Work in | What you do |
| --- | --- | --- | --- |
| 1. Testing | 30 min | pairs/small groups | Write tests for small pipeline functions, then write data checks for a silver table |
| 2. Version control | 15 min | individually | Run git from the notebook: commit, inspect a change, find a bug and undo it |

**Why this matters.** In the lecture we listed what goes wrong when analytics code lives only in a notebook: it is run in the wrong order, it has hidden assumptions, and nobody knows what changed. Two practices address most of that:

- **Tests** turn "I think this works" into evidence that is repeatable and can be run by a machine.
- **Version control** records every change, who made it and why, so a mistake can be found and undone.

You will use the card-spend data from the pipeline practice task, so there is no new dataset to learn.

**Two things to know before you start**
- You are not writing a pipeline here. The functions are provided; your job is to decide **what they should do** and prove it.
- Part 2 uses git commands typed into notebook cells with a leading `!`. Git is a tool installed on the Colab machine; GitHub is a website that stores a copy of a repository. Here you use git only.

In [ ]:
# ---- Setup: run this cell first ----
!pip install -q pytest

import glob
import os
import subprocess

import pandas as pd

# Instructor: set DATA_URL to the link of the card-spend practice zip.
DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/testgitsession/card-spend-practice.zip"
DATA_DIR = "/content/card-spend"

# Download only if no CSV files are there yet. The CSVs may sit inside a folder within the zip.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/card-spend.zip")
    zipfile.ZipFile("/content/card-spend.zip").extractall(DATA_DIR)


def find(filename):
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if not found:
        have = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
        raise FileNotFoundError(f"Missing {filename}. CSV files found under {DATA_DIR}: {have}. Ask for help.")
    return found[0]


bronze_tx = pd.read_csv(find("card_transactions.csv"), dtype={"ref_txn_id": "string"})
bronze_cu = pd.read_csv(find("customer_master.csv"))

WORK = os.getcwd()   # the folder where this notebook works; Part 2 returns to it
print(f"bronze_tx: {bronze_tx.shape[0]:,} rows | bronze_cu: {bronze_cu.shape[0]:,} rows | working folder: {WORK}")

---
# Part 1. Testing

## 1.1 What a test is

A test is a small piece of code that calls a function with an input you chose and checks the result against what you expect. If the result is different, the test fails and tells you.

That sounds trivial, but it forces a decision that is easy to skip: **what should this function do?** In a notebook that decision stays in your head. In a test it is written down.

Run the next cell. It shows the raw currency codes in the bronze data.

In [ ]:
%cd {WORK}
bronze_tx["currency"].value_counts()

Lower-case codes and trailing spaces. A function that cleans these should be *proved* to work, not assumed to.

Below is the module with the functions you will test. **Read it, but do not edit it.** It is written to a file, `pipeline.py`, because that is how real code is shared: one file, many callers.

In [ ]:
%%writefile pipeline.py
"""The above magic function ("%%writefile") means we are creating a .py script in this cell"""

"""Small, pure functions for the card-spend pipeline (silver layer)."""
import pandas as pd

# Later statuses outrank earlier ones when two rows share a file date.
RANK = {"PENDING": 0, "POSTED": 1, "REVERSED": 2}


def clean_currency(series):
    """
    Strip spaces and upper-case currency codes.
    INPUT: a Series of data
    OUTPUT: Returns a new Series.
    """
    return series.str.strip().str.upper()


def parse_timestamp(text):
    """
    Read a processor timestamp. Two formats arrive:
    ISO 'YYYY-MM-DD HH:MM:SS' and day-first 'DD/MM/YYYY HH:MM' (contains a '/').
    Anything else raises ValueError.
    INPUT: A date value as string
    OUTPUT: A date value as datetime (or an error)
    """
    fmt = "%d/%m/%Y %H:%M" if "/" in text else "%Y-%m-%d %H:%M:%S"
    return pd.to_datetime(text, format=fmt)


def latest_status(df):
    """
    Return one row per txn_id: the row from the latest file_date.
    If two rows share a file_date, the higher-ranked status wins (see RANK).
    Exact duplicate rows are removed first. The input is not changed.
    INPUT: A dataframe
    OUTPUT: A modified dataframe
    """
    d = df.drop_duplicates().copy()
    d["_rank"] = d["status"].map(RANK)
    d = d.sort_values(["txn_id", "file_date", "_rank"])
    return d.groupby("txn_id").tail(1).drop(columns="_rank").reset_index(drop=True)


def gbp_amount(amount, currency, rates):
    """
    Convert an amount to GBP. `rates` maps currency -> GBP value of one unit.
    GBP is returned unchanged. An unknown currency raises KeyError (it is never guessed).
    INPUT: Amount, currency, rates
    OUTPUT: Amount in GBP
    """
    if currency == "GBP":
        return amount
    return amount * rates[currency]


def build_silver(bronze):
    """
    Bronze transactions -> silver: duplicates removed, codes cleaned, timestamps parsed, latest status kept.
    INPUT: A dataframe (bronze)
    OUTPUT: A modified dataframe (silver)
    """
    d = bronze.drop_duplicates().copy()
    d["currency"] = clean_currency(d["currency"])
    d["ts"] = d["txn_timestamp"].map(parse_timestamp)
    return latest_status(d)

## 1.2 Write the unit tests

The next cell writes a second file, `tests.py`. A test is a function whose name starts with `test_`, containing `assert` statements. An `assert` passes silently if the condition is true and fails loudly if not.

- The first test is done for you as a **worked example**.
- For each remaining test, the docstring tells you the **situation**. You decide **what the correct answer is** and write the `assert`.
- Build small tables with the helper `make_rows(...)`. You do not need the real data for a unit test; tiny, hand-made inputs are the point.
- Aim to finish **`parse_timestamp` and `latest_status`**. `gbp_amount` is the stretch.

Unfinished tests raise `NotImplementedError`, so they show as **failed** until you write them. Re-run the cell whenever you change it.

*Hint: `pytest.raises(ValueError)` checks that code raises an error, and `pytest.approx(x)` compares decimals safely.*

In [ ]:
%%writefile tests.py
import pandas as pd
import pytest

from pipeline import clean_currency, parse_timestamp, latest_status, gbp_amount


def make_rows(*rows):
    """Build a small bronze table. Each row is (txn_id, status, file_date, amount)."""
    return pd.DataFrame(rows, columns=["txn_id", "status", "file_date", "amount"])


# ---------------- worked example ----------------
def test_clean_currency_fixes_case_and_spaces():
    raw = pd.Series(["gbp", "EUR ", " usd", "GBP"])
    assert clean_currency(raw).tolist() == ["GBP", "EUR", "USD", "GBP"]

'''
COMMENTARY ON THE EXAMPLE
We can see that the raw sequence ["gbp", "EUR ", " usd", "GBP"] would fail if there
were no function as they do not match the expected output - ["GBP", "EUR", "USD", "GBP"].
We call the clean_currency function on raw and it should change it to upper case and remove the spaces.
This should mean we pass the test now.
The 'assert' commands basically means test whether the function output matches the expected output.
The test should pass
'''


# ---------------- parse_timestamp ----------------
def test_parse_iso():
    """An ISO timestamp, '2026-03-08 13:25:00', is read correctly."""
    raise NotImplementedError("write this test")

def test_parse_day_first():
    """'08/03/2026 13:25' arrives from the processor. Which date is it? Decide, then assert."""
    raise NotImplementedError("write this test")


# ---------------- latest_status ----------------
def test_later_file_wins():
    """T1 is PENDING in the 5 Jan file and POSTED in the 6 Jan file. One row should remain. Which status?"""
    raise NotImplementedError("write this test")


def test_same_file_prefers_posted():
    """T1 appears as POSTED and PENDING in the SAME file (list POSTED first, on purpose). Which status wins?"""
    raise NotImplementedError("write this test")


def test_exact_duplicates_are_removed():
    """The same row arrives twice (a re-sent file). How many rows should remain?"""
    raise NotImplementedError("write this test")


# ---------------- gbp_amount (stretch - if you have time) ----------------
def test_gbp_is_unchanged():
    """10.0 GBP stays 10.0."""
    raise NotImplementedError("write this test")


def test_eur_is_converted():
    """100.0 EUR at a rate of 0.85 is worth how many GBP?"""
    raise NotImplementedError("write this test")


def test_unknown_currency_is_not_guessed():
    """5.0 JPY with no JPY rate available. What should happen, and why is guessing a rate dangerous?"""
    raise NotImplementedError("write this test")

Now run the tests. `-q` keeps the output short.

In [ ]:
!python -m pytest -q tests.py

Read the result line at the bottom (for example `3 failed, 7 passed`). If a test fails, read the message before changing anything: *is the test wrong, or is the function wrong?* Either is possible. Deciding which is the real skill.

**Checkpoint.** You should have at least the `parse_timestamp` and `test_parse_iso` tests passing. Keep your `tests.py` as it is: you will use it in Part 2.

## 1.3 Data tests on a silver table

Unit tests check functions on tiny inputs. Data tests check the actual table a pipeline has produced. They catch problems that no one thought to put in a unit test, such as a new kind of bad row arriving next month.

The standard data tests have names you will meet again in the module and in practice: **unique**, **not null**, **accepted values** and **relationships**.

Every check needs a decision as well as a condition: *what happens when it fails?*

| Action | Meaning |
| --- | --- |
| `fail` | Stop the pipeline. The problem is serious enough that nothing downstream can be trusted. |
| `quarantine` | Move the bad rows aside for someone to review; carry on with the rest. |
| `flag` | Keep the rows, mark them, and carry on. |

First build the silver table with the provided function:

In [ ]:
import importlib
import sys

# Add the working folder to the front of Python's module search path,
# so "import pipeline" finds the pipeline.py file that was written there.
# WORK was defined in the setup cell (the folder the notebook works in).
sys.path.insert(0, WORK)

import pipeline    # load pipeline.py as a module, so its functions can be called as pipeline.<name>

# Python remembers modules it has already imported. If pipeline.py has been
# rewritten since the first import, this forces a fresh read of the file.
importlib.reload(pipeline)

# Run the whole bronze-to-silver step on the raw transactions. The result is a DataFrame.
silver = pipeline.build_silver(bronze_tx)

# Print how many rows silver has, and how many bronze had.
# In the f-string, {x:,} formats a number with thousands separators (e.g. 15126 becomes 15,126).
print(f"silver: {len(silver):,} rows (from {len(bronze_tx):,} bronze rows)")


# Define a helper that reports how many rows break a rule and what the pipeline would do.
# name: a label for the check.
# bad_rows: the rows that failed the check.
# action: what to do when rows fail ("fail", "quarantine" or "flag").
def run_check(name, bad_rows, action):
    # A docstring: a description of what the function is for.
    """Report how many rows break a rule, and what the pipeline would do about it."""

    # A dictionary mapping each allowed action to the message printed when rows fail.
    outcomes = {"fail": "PIPELINE STOPS", "quarantine": "rows set aside", "flag": "rows kept and flagged"}

    # If the action isn't one of the three keys (for example "___", the blank
    # students start with), print a prompt to choose one...
    if action not in outcomes:
        # {name:<45} pads the name to 45 characters, left-aligned, so columns line up.
        print(f"{name:<45} choose an action: fail, quarantine or flag")
        # ...and leave the function early. A bare return gives back None.
        return

    # Count the failing rows.
    n = len(bad_rows)

    # Print the name, the count right-aligned in 6 characters ({n:>6}), and then
    # "ok" if there were no failures, otherwise the message for the chosen action.
    print(f"{name:<45} {n:>6} rows  {'ok' if n == 0 else outcomes[action]}")

**Check 1 is a worked example.** In each of the cells below, replace `YOUR_CONDITION_HERE` with a condition that selects the **bad** rows, and replace `___` with an `action` (remember this can be `fail`, `quarantine` or `flag`). Do **checks 2 and 3 first**; checks 4 and 5 are the stretch. Be ready to justify each choice in one sentence.

In [ ]:
# Check 1 (worked example): txn_id is unique
bad = silver[silver["txn_id"].duplicated(keep=False)]
run_check("txn_id is unique", bad, action="fail")

In [31]:
# Check 2: currency is one of the accepted values (GBP, EUR, USD, CHF)
ACCEPTED = ["GBP", "EUR", "USD", "CHF"]
bad = silver[YOUR_CONDITION_HERE]
run_check("currency is an accepted value", bad, action="___")

currency is an accepted value                      0 rows  ok


In [ ]:
# Check 3 (stretch - if you have time): sign rule. REVERSED rows must be negative; every other row must not be negative.
bad = silver[YOUR_CONDITION_HERE]
run_check("amount sign matches status", bad, action="___")

In [ ]:
# Check 4 (stretch): every card_id exists in the customer master (a 'relationships' test)
bad = silver[YOUR_CONDITION_HERE]
run_check("card_id exists in customer master", bad, action="___")

**Justify your choices** (one sentence each). Think about what is lost if the pipeline stops, what is lost if bad rows slip through, and who would need to look at quarantined rows.

1. Currency:
2. Sign rule:
3. Unknown cards:

**Pair discussion (2 minutes):** your unit tests passed on tiny hand-made tables. Which of the five data checks would catch a problem that no unit test of yours would have caught, and why?

---

# Part 2. Version control with git (30 minutes, individually)

## What git does

Git keeps a history of your files as a series of commits. Each commit is a snapshot plus a message saying what changed and why. You can compare any two snapshots, and you can undo a change without losing the record that it happened.

**Git is not GitHub.** Git records history on the machine where you run it. GitHub hosts a copy so you can share it and keep it safe.

> **Important:** in this workshop you run git inside Colab. Colab's machine is temporary, so commits made here disappear when the session ends. That is fine for learning the commands. At the end you save this notebook to GitHub with its outputs, which keeps your command history as evidence. Publishing commits to GitHub with `git push` needs a credential and is not part of this exercise.

## 2.1 Get a repository to work in

Edit the three lines below, then run the cell. It clones your repository if it can; otherwise it creates a new empty one so you can carry on.

- **Use your GitHub no-reply address**, not your real email. Commits record an email address, and your repository is public. You can find the no-reply address on GitHub under *Settings, then Emails*.
- `REPO_URL` is the web address of your repository followed by `.git`.

**DON'T WORRY ABOUT THE REST OF THE CODE. IT'S JUST TO MAKE THE CODE WORK IN COLAB.**

In [ ]:
# ---- Your details: edit these three lines ----
REPO_URL = "YOUR REPO URL FOLLOWED BY .git - e.g. https://github.com/MJMortensonWarwick/AnalyticsEngineering2026.git"
GIT_NAME = "YOUR NAME"
GIT_EMAIL = "EMAIL"   # your GitHub no-reply address (see above), not your real email

REPO_DIR = REPO_URL.rstrip("/").split("/")[-1]
if REPO_DIR.endswith(".git"):
    REPO_DIR = REPO_DIR[:-4]
REPO_PATH = os.path.join(WORK, REPO_DIR)

if not os.path.isdir(REPO_PATH):
    r = subprocess.run(["git", "clone", REPO_URL, REPO_PATH], capture_output=True, text=True,
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode == 0:
        print("Cloned your repository.")
    else:
        print("Could not clone", REPO_URL)
        print(r.stderr.strip()[:300])
        print("Creating a fresh local repository instead. Everything below still works.")
        subprocess.run(["git", "init", "-b", "main", REPO_PATH], check=True, capture_output=True)

%cd {REPO_PATH}
!git config user.name "{GIT_NAME}"
!git config user.email "{GIT_EMAIL}"
MAIN = subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True).stdout.strip()
print("Working in:", os.getcwd(), "| branch:", MAIN)

## 2.2 Your first commit

Copy your two files from Part 1 into a folder in the repository, and check the tests still pass from there.

In [ ]:
!mkdir -p week3-testing
!cp {WORK}/pipeline.py {WORK}/tests.py week3-testing/
%cd week3-testing
!python -m pytest -q tests.py

Git works in three steps: **change** files, **stage** the ones you want to record (`git add`), then **commit** them with a message (`git commit -m "..."`). `git status` shows where you are.

In [ ]:
!git add pipeline.py tests.py
!git status

`git status` lists something you did **not** stage: a `__pycache__` folder that Python created when it ran your tests. That is generated output, not your work, and it should never be committed. A file named `.gitignore` tells git to ignore such things.

In [ ]:
%%writefile .gitignore
__pycache__/
.pytest_cache/

Stage `.gitignore` too, then commit. **Write your own message.** A useful message says *what changed and why*, in the first line, in the imperative:

| Poor | Better |
| --- | --- |
| `update` | `Add pipeline functions and unit tests for timestamps and latest status` |
| `stuff` | `Ignore Python cache folders` |
| `final version` | `Fix day-first timestamp parsing` |

In [ ]:
!git add .gitignore
!git commit -m "___"
!git log --oneline

## 2.3 Keep a record (about 3 minutes)

Print your history in a compact form. This output will be saved inside the notebook, so it becomes evidence of what you did. (The format deliberately leaves out email addresses.)

In [ ]:
!git log --format="%h  %ad  %an  %s" --date=short

**Now save this notebook to your GitHub repository:** in Colab choose *File, then Save a copy in GitHub*, pick your repository, and write a commit message that says what the notebook contains. The outputs, including the history above, are saved with it.

### Stretch: branches (only if you have time)

A **branch** is a separate line of work. You can try something, and either merge it back or throw it away, without touching the main line. Fill in the blanks:

In [ ]:
!git switch -c add-gbp-note
!echo "# gbp_amount raises KeyError for unknown currencies" >> tests.py
!git add tests.py
!git commit -m "Note the unknown-currency rule in the tests file"
!git switch {MAIN}
!git ___ add-gbp-note
!git log --oneline

### Warning: publishing to GitHub

`git push` sends your commits to GitHub. It needs a **personal access token**, which is a password-like secret. **Never paste a token into a notebook cell**: your repository is public, so anyone could use it.